In [12]:
import os
import pandas as pd
import numpy as np
from src.common.utils import get_root_directory, collate_array_elements, gridsearch
import matplotlib.pyplot as plt
from src.preprocessing.data_loader import DataLoader
from src.common.stats import root_mean_squared_percentage_error
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, precision_score, accuracy_score
import mlflow
import mlflow.sklearn
from xgboost import XGBRegressor
import shap
from sklearn.preprocessing import MinMaxScaler

In [13]:
#PARAMETERS TO CHANGE
model_type = "XGB"

n_periods = 7
start_date = "01/01/2021"
end_date = "31/12/2023"
split_size = 0.15

selected_data = []
sentiment_type = "cryptobert"
ETH_base = ['ETH_D_AvgPrc']
ETH_blk = False
ETH_features = ['ETH_D_PrcDir','ETH_YF_Vol','ETH_ES_AvgTransFee','ETH_ES_BlkTm','ETH_ES_GasUsd','ETH_ES_AvgGasPrc']
BTC_features = ['BTC_D_AvgPrc','BTC_OL_MinDif','BTC_OL_AvgTransFee','BTC_BIC_HshRt']
LTC_features = ['LTC_D_AvgPrc','LTC_YF_Vol','LTC_OL_MinDif','LTC_BIC_HshRt']
sentiment_features = ['D_VADER_AvgScr_Ex', 'D_VADER_Sent_AvgEx','D_FINBERT_AvgScr_Ex','D_FINBERT_Sent_AvgEx','D_CRYPTOBERT_AvgScr_Ex','D_CRYPTOBERT_Sent_AvgEx']
filtered_features = ETH_base
run_name = "ETH_base"
if ETH_blk:
    filtered_features.extend(ETH_features)
    run_name = run_name +"_ETH_blk"
if 'BTC' in selected_data:
    filtered_features = filtered_features + BTC_features
    run_name = run_name +"_BTC_blk"
if 'LTC' in selected_data:
    filtered_features = filtered_features + LTC_features
    run_name = run_name +"_LTC_blk"
if 'sentiment_analysis' in selected_data:
    filtered_features = filtered_features + [feature for feature in sentiment_features if sentiment_type.upper() in feature]
    run_name = run_name +f"_sent_{sentiment_type}"
    
random_state=42

split_name = "test"
experiment_name = f"{model_type.upper()}_train_{split_name}"

In [14]:
param_grid = {
    "lags": [7],
    "gamma": [5],
    "reg_lambda":[4],
    "reg_alpha": [3],
    "n_estimators":[100]
}

In [15]:
@gridsearch(param_grid)
def run_experiment(lags, gamma, reg_lambda, reg_alpha, n_estimators):
    root_dir = get_root_directory()
    DL = DataLoader(root_dir)
    DL.load_data()
    DL.merge_selected_data(selected_data=selected_data)
    DL.select_features(features=filtered_features)
    lag_config = {col:lags for col in filtered_features}
    DL.lag_features(lag_config=lag_config)
    DL.generate_forecast_horizon(horizon=n_periods)
    DL.set_time_range(start_date=start_date, end_date=end_date)
    if split_name=="test":
        train, test = DL.split_data(split_size=split_size)
        split = test
    elif split_name=="validation":
        train, test = DL.split_data(split_size=split_size)
        train, val = DL.split_data(split_type="train_val", split_size=split_size)
        split = val
    features = [col for col in train.columns if 'lag' in col]
    targets = [col for col in train.columns if'horizon' in col]
    train = train[:-n_periods]
    split = split[:-n_periods]
    y_scaler = MinMaxScaler()
    x_scaler = MinMaxScaler()
    y_train, x_train = y_scaler.fit_transform(train[targets].to_numpy()), x_scaler.fit_transform(train[features].to_numpy())
    y_val, x_val = split[targets].to_numpy(), x_scaler.transform(split[features].to_numpy())
    mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/{model_type.upper()}/mlruns.db")
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "lags":lags,
            "n_estimators": n_estimators,
            "gamma": gamma,
            "reg_lambda": reg_lambda,
            "reg_alpha":reg_alpha,
            "n_periods": n_periods,
            "prediction_start_date":split.index.to_list()[0].strftime('%d-%m-%Y'),
            "features":filtered_features
        })
        XGB = XGBRegressor(random_state=42, multi_strategy='multi_output_tree', gamma = gamma, reg_lambda = reg_lambda, reg_alpha = reg_alpha, n_estimators=n_estimators)
        XGB.fit(x_train, y_train)
        df = pd.DataFrame(y_scaler.inverse_transform(XGB.predict(x_val)), columns=[f"t{i+1}" for i in range(n_periods)])
        mse_list = []
        rmse_list = []
        rmspe_list = []
        mae_list = []
        mape_list = []
        accuracy_list  = []
        precision_list = []
        for i in range(n_periods):
            df[f't{i+1}_prc_dir'] = df[f't{i+1}'].diff().apply(lambda x: 1 if x > 0 else -1)
            mse = mean_squared_error(y_val[:,i], df[f't{i+1}'])
            rmse = root_mean_squared_error(y_val[:,i],  df[f't{i+1}'])
            rmspe = root_mean_squared_percentage_error(y_val[:,i],  df[f't{i+1}'])
            mae = mean_absolute_error(y_val[:,i],  df[f't{i+1}'])
            mape = mean_absolute_percentage_error(y_val[:,i],  df[f't{i+1}'])
            accuracy = accuracy_score(pd.Series(y_val[:,i]).diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
            precision = precision_score(pd.Series(y_val[:,i]).diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
            mse_list.append(mse)
            rmse_list.append(rmse)
            rmspe_list.append(rmspe)
            mae_list.append(mae)
            mape_list.append(mape)
            accuracy_list.append(accuracy)
            precision_list.append(precision)
            avg_mse = sum(mse_list)/len(mse_list)
            avg_rmse = sum(rmse_list)/len(rmse_list)
            avg_rmspe = sum(rmspe_list)/len(rmspe_list)
            avg_mae = sum(mae_list)/len(mae_list)
            avg_mape = sum(mape_list)/len(mape_list)
            avg_accuracy = sum(accuracy_list)/len(accuracy_list)
            avg_precision = sum(precision_list)/len(precision_list)
            mlflow.log_metrics({
                        "mse_daily":mse_list[i], 
                        "rmse_daily":rmse_list[i],
                        "rmspe_daily":rmspe_list[i], 
                        "mae_daily":mae_list[i], 
                        "mape_daily":mape_list[i], 
                        "accuracy_daily":accuracy_list[i],
                        "precision_daily":precision_list[i],
                        "avg_mse": avg_mse,
                        "avg_rmse": avg_rmse,
                        "avg_rmspe": avg_rmspe,
                        "avg_mae": avg_mae,
                        "avg_mape": avg_mape,
                        "avg_accuracy":avg_accuracy,
                        "avg_precision":avg_precision
                    }, step=(i+1))
    mlflow.end_run()
    if split_name=="test":
        results_file = f"{run_name}_{split_name}"
        path = str(os.path.join(root_dir,"mlruns",model_type))
        df.to_csv(str(os.path.join(path,results_file)), index=False)

In [16]:
run_experiment()

Running with params: {'lags': 7, 'gamma': 5, 'reg_lambda': 4, 'reg_alpha': 3, 'n_estimators': 100}


2025/08/10 15:59:11 INFO mlflow.tracking.fluent: Experiment with name 'XGB_train_test' does not exist. Creating a new experiment.


In [10]:
from mlflow.tracking import MlflowClient
import pandas as pd

# Config
root_dir = get_root_directory()
tracking_uri = f"sqlite:///{root_dir}/mlruns/mlruns.db"
experiment_name = experiment_name
metric_name = "rmspe_daily"
avg_metric_name = "avg_rmspe"

# Init MLflow client
client = MlflowClient(tracking_uri=tracking_uri)

# Get experiment ID from name
experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    raise ValueError(f"Experiment '{experiment_name}' not found.")
experiment_id = experiment.experiment_id

# Search all runs in experiment
runs = client.search_runs([experiment_id],max_results=5000)

# Collect metric history from all runs
records = []
for run in runs:
    run_id = run.info.run_id
    history = client.get_metric_history(run_id, metric_name)
    for m in history:
        records.append({
            "run_id": run_id,
            "step": m.step,
            "value": m.value
        })

# Create DataFrame
df = pd.DataFrame(records)

# Pivot so each step is a column
df_pivot = df.pivot(index="run_id", columns="step", values="value")
df_pivot = df_pivot.apply(pd.to_numeric, errors="coerce")

# Compute average per run
df_pivot["average"] = df_pivot.mean(axis=1, skipna=True)

# Log average metric back to MLflow for each run
for run_id, avg_value in df_pivot["average"].items():
    client.log_metric(run_id, avg_metric_name, float(avg_value))

print("Averages logged back to MLflow successfully.")

Averages logged back to MLflow successfully.
